[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/11_aprendizaje_contrastivo/11_aprendizaje_contrastivo.ipynb)
# Módulo 11 · Aprendizaje contrastivo para embeddings

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración** | ≈ 6 h (lección ≈ 3.5 h + proyecto ≈ 2.5 h) |
| **Cómputo** | `FAST_DEV_RUN=True`: CPU o T4, ≈ 10–12 min (modelos diminutos). Modo completo: **L4 o A100 recomendada** (ver la caja «Estimación de cómputo» para calcular unidades de Colab) |
| **Prerrequisitos** | 09 (capa de embedding), 10 (pooling, bi-encoders, anisotropía); softmax y entropía cruzada |
| **Hilo conductor** | Políglota entrena **su propio embedder** ES/EN/código: el *baseline* denso del módulo 10 era un modelo ajeno; hoy construyes el tuyo |

> **Dónde estamos.** En el 10 vimos que un encoder sin entrenar para similitud da embeddings pobres y que el post-procesado no crea señal. La solución industrial es **entrenar** el bi-encoder con un objetivo **contrastivo**: acercar pares relevantes y alejar los no relevantes. Lo implementas desde cero (InfoNCE, bi-encoder, Gradient Cache) y luego con `sentence-transformers`.

## Objetivos de aprendizaje
1. **Escribir** InfoNCE con cada símbolo explicado y **derivar** el papel de la temperatura en el gradiente.
2. **Derivar** por qué InfoNCE es una cota inferior de la información mutua, $I(X;Y)\ge\log K-\mathcal L$, y **comprobarlo numéricamente** (incluida la saturación en $\log K$).
3. **Implementar desde cero** en PyTorch un bi-encoder y su entrenamiento con negativos en el lote, negativos duros y pérdida simétrica.
4. **Diagnosticar** colapso y falsos negativos con las métricas de **alineación y uniformidad** (Wang & Isola, 2020).
5. **Minar** negativos duros con BM25 y **filtrar** falsos negativos (TopK-PercPos, NV-Retriever).
6. **Implementar** Gradient Cache y **verificar** que reproduce el gradiente del lote completo.
7. **Entrenar** con `sentence-transformers` (`MultipleNegativesRankingLoss`) y **calcular** cuántos parámetros entrena LoRA.

## 🔁 Conexión con módulos anteriores
<details><summary><b>1.</b> (10) ¿Por qué el producto escalar entre embeddings normalizados es «indexable» y una puntuación de cross-encoder no?</summary>

Porque $\cos(f(q), f(d))$ factoriza: $f(d)$ se calcula una vez por documento y se indexa; la búsqueda es un producto con todos los vectores (o ANN). Un cross-encoder necesita un forward por par.
</details>

<details><summary><b>2.</b> (09) La cabeza de un LM es $\text{softmax}(W_U\mathbf h)$ sobre $V$ tokens. ¿Qué tiene que ver con recuperar un documento entre $N$?</summary>

Es el mismo problema con «tokens» = documentos: $p(d\mid q)=\text{softmax}_d(f(q)\cdot f(d)/\tau)$. Con $N$ = millones, el softmax completo es imposible y se **muestrea**: los demás documentos del lote hacen de negativos. En el 09 viste también que los ítems que nunca son positivos solo reciben gradiente «de negativo».
</details>

<details><summary><b>3.</b> (10) ¿Por qué un modelo entrenado con *mean pooling* no debe usarse con CLS?</summary>

Porque el entrenamiento ajustó el espacio que produce **ese** pooling; otro pooling es otra función que nunca se optimizó.
</details>

<details><summary><b>4.</b> (00, Diagnóstico) Usarás el top de BM25 (módulo 00) como «negativos duros»: documentos parecidos a la consulta que no están en los qrels. ¿Por qué algunos de ellos serán, en realidad, relevantes?</summary>

Porque los qrels del golden set son **incompletos**: se anotó lo que alguien juzgó, no todo el corpus. Un documento con mucho solapamiento léxico y sin juicio puede ser relevante; tratarlo como negativo enseña al modelo a alejar lo que debería acercar (falso negativo, sección 5). Es el mismo problema de los «no clics» en recsys: no visto ≠ no relevante.
</details>

In [ ]:
!pip install -q transformers sentence-transformers peft accelerate tokenizers datasets scipy scikit-learn matplotlib
import copy, json, math, os, random, re, time, warnings, zlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
import transformers
transformers.logging.set_verbosity_error(); transformers.utils.logging.disable_progress_bar()
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
FAST_DEV_RUN = True   # True: CPU, modelos diminutos, ~10 min. False: más pasos/datos (GPU L4/A100 recomendada)
PAL = {"azul": "#1F6FEB", "naranja": "#F28E2B", "verde": "#2CA02C", "gris": "#6B7280", "morado": "#9467BD"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · torch {torch.__version__}")

### Utilidades del curso: corpus Políglota, métricas y BM25 canónicos (módulo 00)

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
# ── Tokenización simple + BM25 propio (módulo 00) ─────────────────────────────
import math
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np

_WORD_RE = re.compile(r"\w+", re.UNICODE)


def fold_accents(text: str) -> str:
    """'canción' -> 'cancion' (NFD y descartar marcas combinantes Mn)."""
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if unicodedata.category(c) != "Mn"))


def tokenize_simple(text: str, accents: str = "keep") -> list[str]:
    """Minúsculas + palabras \\w+. accents='fold' quita tildes. (Esto NO es el 'token' de un LLM: es un token léxico.)"""
    text = unicodedata.normalize("NFC", text).lower()
    if accents == "fold":
        text = fold_accents(text)
    return _WORD_RE.findall(text)


class BM25:
    """BM25 con índice invertido. idf='lucene': ln(1+(N-n+.5)/(n+.5)) (siempre >= 0).
    idf='okapi': ln((N-n+.5)/(n+.5)) con suelo epsilon*idf_medio (como rank_bm25.BM25Okapi)."""

    def __init__(self, k1: float = 1.2, b: float = 0.75, idf: str = "lucene", epsilon: float = 0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens: list[list[str]], doc_ids: list[str]) -> "BM25":
        self.doc_ids = list(doc_ids)
        self.N = len(docs_tokens)
        self.dl = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avgdl = float(self.dl.mean()) if self.N else 0.0
        self.postings = defaultdict(list)  # término -> [(índice_doc, tf)]
        for i, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        raw = {t: len(p) for t, p in self.postings.items()}
        if self.idf_kind == "lucene":
            self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in raw.items()}
        else:
            self.idf = {t: math.log(self.N - n + 0.5) - math.log(n + 0.5) for t, n in raw.items()}
            floor = self.epsilon * sum(self.idf.values()) / len(self.idf)
            self.idf = {t: (v if v >= 0 else floor) for t, v in self.idf.items()}
        return self

    def scores(self, q_tokens: list[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for term in q_tokens:  # los términos repetidos de la query suman varias veces (igual que rank_bm25)
            for i, tf in self.postings.get(term, ()):
                norm = self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / (tf + norm)
        return s

    def search(self, q_tokens: list[str], k: int = 10) -> list[str]:
        s = self.scores(q_tokens)
        order = np.lexsort((np.arange(self.N), -s))[:k]  # empates -> orden del corpus
        return [self.doc_ids[i] for i in order]

In [ ]:
corpus = carga_corpus(FAST_DEV_RUN)
docs, queries, qrels = corpus["docs"], corpus["queries"], corpus["qrels"]
doc_ids = [d["id"] for d in docs]
doc_txt = [d["text"] for d in docs]
txt_of = dict(zip(doc_ids, doc_txt))
# partición determinista de las consultas: ~50 % entrenamiento / ~50 % test
split_q = {q["qid"]: ("train" if zlib.crc32(q["qid"].encode()) % 2 == 0 else "test") for q in queries}
q_train = [q for q in queries if split_q[q["qid"]] == "train"]
q_test = [q for q in queries if split_q[q["qid"]] == "test"]
print("fuente:", corpus["source"], "| docs:", len(docs), "| queries train/test:", len(q_train), len(q_test))

## 1. Intuición: recuperar = clasificar entre todos los documentos

Si tuviéramos un único corpus fijo, podríamos entrenar un clasificador «consulta → id de documento» con softmax sobre los $N$ documentos:
$$p(d\mid q)=\frac{\exp\big(s(q,d)/\tau\big)}{\sum_{d'=1}^{N}\exp\big(s(q,d')/\tau\big)},\qquad s(q,d)=f(q)\cdot f(d).$$
Dos problemas: $N$ es enorme y cambia (documentos nuevos). La solución contrastiva: aproximar el denominador con **unos pocos documentos** — el positivo y $K-1$ negativos — y, sobre todo, **reusar como negativos los positivos de las demás consultas del lote** (*in-batch negatives*): $B$ pares dan $B(B-1)$ negativos «gratis» con una sola matriz $B\times B$.

> 💡 **Recsys.** Es exactamente la *softmax muestreada con negativos en el lote* del *two-tower* (`recsys-course/08_deep_retrieval_ann`), incluido su sesgo: los ítems populares aparecen más como negativos (allí se corrige con **logQ**; en texto es menos habitual porque los documentos positivos están mucho más repartidos). Lo nuevo aquí es el codificador (un Transformer) y el tipo de pares (consulta–pasaje, título–cuerpo, docstring–código…).

> ### 🎨 GEMINI-IMAGEN · ID: `M11-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Anatomía de un embedder contrastivo**
>
> **Qué muestra:** Muestra el pipeline completo de entrenamiento contrastivo: pares, torre compartida, normalización, matriz de similitud del lote y pérdida InfoNCE.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Pipeline contrastivo: 4 pares consulta-documento pasan por un encoder compartido; la matriz 4×4 de similitudes tiene la diagonal como positivos y la pérdida es entropía cruzada por filas.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9, sin decoración. Paleta: azul #1F6FEB (consultas), naranja #F28E2B (documentos), verde #2CA02C (positivos), gris #6B7280 (negativos y texto secundario). Tipografía sans-serif.
> Izquierda: una columna de 4 cajas azules «q₁: cómo configurar la caché», «q₂: what is a hash table», «q₃: limitación de tasa», «q₄: unit tests» y otra columna de 4 cajas naranjas «d₁: La caché es una memoria rápida…», «d₂: A hash table stores key-value pairs…», «d₃: Una política que restringe peticiones…», «d₄: Automatic checks that verify a function…». Líneas verdes finas unen qᵢ con dᵢ con la etiqueta «pares positivos».
> Centro: ambas columnas entran en UNA torre rotulada «Encoder compartido (Transformer + mean pooling)» y salen vectores con la etiqueta «normalizar ‖·‖ = 1».
> Derecha: una matriz 4×4 rotulada «S = Q·Dᵀ / τ» con la diagonal en verde («positivos») y el resto en gris («negativos del lote»). A la derecha de la matriz, una flecha hacia la caja «InfoNCE = entropía cruzada por filas con etiqueta = diagonal».
> Pie: «B pares → B·(B−1) negativos gratis».
> Evita: texto ilegible, flechas ambiguas, números inventados, iconos decorativos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M11-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Pipeline contrastivo: 4 pares consulta-documento pasan por un encoder compartido; la matriz 4×4 de similitudes tiene la diagonal como positivos y la pérdida es entropía cruzada por filas.](assets/M11-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> q1..q4 ─┐                              S = Q·Dᵀ/τ   d1  d2  d3  d4
>         ├─► [Encoder compartido] ─► ‖·‖=1 ─► q1 [ +   -   -   - ]
> d1..d4 ─┘                                  q2 [ -   +   -   - ]   InfoNCE = CE por filas
>                                            q3 [ -   -   +   - ]   (etiqueta = diagonal)
>                                            q4 [ -   -   -   + ]
> ```

> 👀 **Qué debes observar.** Una sola torre (pesos compartidos) para consultas y documentos, vectores normalizados, y la pérdida es **solo una entropía cruzada** cuya «clase correcta» es la diagonal: todo el aprendizaje contrastivo de esta lección son variaciones de qué se pone en las columnas.

## 2. 📐 InfoNCE, temperatura e información mutua

### Definición
Para un lote de $B$ pares $(q_i, d_i)$ con embeddings normalizados $\mathbf u_i=f(q_i)$, $\mathbf v_j=f(d_j)$:
$$\mathcal L_{\text{InfoNCE}}=-\frac1B\sum_{i=1}^{B}\log\frac{\exp(\mathbf u_i\cdot\mathbf v_i/\tau)}{\sum_{j=1}^{B}\exp(\mathbf u_i\cdot\mathbf v_j/\tau)}.$$
**Símbolos.** $B$ = tamaño del lote; $\mathbf u_i\cdot\mathbf v_j=\cos(q_i,d_j)\in[-1,1]$; $\tau>0$ = **temperatura**; el numerador es el positivo y el denominador suma el positivo y los $B-1$ negativos del lote. Si se añaden negativos duros $\mathbf v^-_m$, entran como columnas extra del denominador.

✍️ **Ejemplo a mano ($B = 3$, una fila).** La consulta $q_1$ tiene cosenos $(0.8,\ 0.5,\ 0.1)$ con $(d_1, d_2, d_3)$; $d_1$ es su positivo.
* Con $\tau = 1$: $e^{0.8}, e^{0.5}, e^{0.1} = 2.23, 1.65, 1.11$ → $p = (0.45, 0.33, 0.22)$ → $\mathcal L_1 = -\ln 0.45 = 0.80$.
* Con $\tau = 0.1$: $e^{8}, e^{5}, e^{1} = 2981, 148, 2.7$ → $p = (0.952, 0.047, 0.001)$ → $\mathcal L_1 = 0.049$.

Con $\tau = 0.1$ el negativo $d_2$ (coseno 0.5, el «duro») recibe 50 veces más empuje que $d_3$; con $\tau = 1$ apenas 1.5 veces más. Es una clasificación de 3 clases (módulo 09: la cabeza del LM hace lo mismo sobre $V$ tokens) y es la *softmax muestreada* de un two-tower de recsys. La sección siguiente lo formaliza.

### Temperatura: qué hace con el gradiente
Con $p_{ij}=\text{softmax}_j(\mathbf u_i\cdot\mathbf v_j/\tau)$:
$$\frac{\partial\mathcal L_i}{\partial(\mathbf u_i\cdot\mathbf v_j)}=\frac1\tau\big(p_{ij}-\mathbb 1[i=j]\big).$$
Cada negativo empuja con peso $p_{ij}\propto e^{s_{ij}/\tau}$: con $\tau$ **pequeña**, casi todo el peso va al negativo **más parecido** (se concentra en los duros); con $\tau$ **grande**, todos pesan parecido y el modelo apenas distingue (Wang & Liu, 2021). Valores publicados: SimCSE $\tau=0.05$; E5 $\tau=0.01$; en `sentence-transformers`, `MultipleNegativesRankingLoss(scale=20)` equivale a $\tau=1/20=0.05$ (según su documentación: `scale = 1/temperature`).

### InfoNCE como cota de la información mutua
Sea $(x,y)\sim p(x,y)$ un par positivo y $y_2,\dots,y_K\sim p(y)$ negativos independientes. Con un «crítico» $g(x,y)=e^{f(x,y)}$ cualquiera, la pérdida con $K$ candidatos es
$\mathcal L_K=-\mathbb E\big[\log\frac{g(x,y_1)}{\sum_{k=1}^{K}g(x,y_k)}\big]$.
1. **Crítico óptimo.** Clasificar cuál de los $K$ es el positivo es un problema de Bayes: el positivo es $y_k$ con probabilidad $\propto\frac{p(y_k\mid x)}{p(y_k)}$, así que el óptimo es $g^*(x,y)\propto\frac{p(y\mid x)}{p(y)}$ (cociente de densidades).
2. **Sustituir** (Oord et al., 2018): $\mathcal L_K^{*}=\mathbb E\log\Big[1+\frac{p(y_1)}{p(y_1\mid x)}\sum_{k\ge2}\frac{p(y_k\mid x)}{p(y_k)}\Big]$.
3. **Aproximar** la suma por $K-1$ veces su esperanza: $\mathbb E_{y\sim p(y)}\big[\frac{p(y\mid x)}{p(y)}\big]=1$, así que $\mathcal L_K^*\approx\mathbb E\log\big[1+(K-1)\frac{p(y_1)}{p(y_1\mid x)}\big]$.
4. **Acotar**: para el cociente típico de un positivo ($p(y_1\mid x)\ge p(y_1)$), $\;1+(K-1)a\ge K a$ con $a=p(y_1)/p(y_1|x)\le1$, luego $\mathcal L_K^*\gtrsim\mathbb E\log\big[K\frac{p(y_1)}{p(y_1\mid x)}\big]=\log K-I(X;Y)$.

La derivación de Oord et al. es heurística en los pasos 3–4; Poole et al. (2019) demuestran la versión rigurosa: **para cualquier crítico**, $I(X;Y)\ \ge\ \log K-\mathcal L_K$.
**Consecuencias.** (a) Minimizar InfoNCE maximiza una cota inferior de la información mutua entre las dos vistas. (b) La cota **nunca supera $\log K$**: con lotes pequeños no puede «ver» más de $\log B$ nats (McAllester & Stratos, 2020, formalizan este límite). (c) Más negativos = cota más ajustada, una motivación para lotes grandes. (d) Ojo: Tschannen et al. (2020) muestran que maximizar MI **no explica por sí solo** la calidad de las representaciones; la cota es una guía, no la causa.

In [ ]:
# Comprobación numérica con una gaussiana bivariante de MI conocida: I = -0.5·log(1 - ρ²)
def infonce_gauss(rho, K, n=4000, seed=SEED):
    """Estima log K - L_K con el crítico ÓPTIMO (log del cociente de densidades), para n anclas y K candidatos."""
    g = np.random.default_rng(seed)
    x = g.standard_normal(n)
    y_pos = rho * x + np.sqrt(1 - rho ** 2) * g.standard_normal(n)
    y = np.concatenate([y_pos[:, None], g.standard_normal((n, K - 1))], 1)       # columna 0 = positivo
    f = -(y - rho * x[:, None]) ** 2 / (2 * (1 - rho ** 2)) + y ** 2 / 2         # log p(y|x) - log p(y)
    L = -(f[:, 0] - np.log(np.exp(f - f.max(1, keepdims=True)).sum(1)) - f.max(1)).mean()
    return np.log(K) - L


rho = 0.99
I_real = -0.5 * np.log(1 - rho ** 2)
Ks = [2, 4, 8, 16, 32, 64, 128, 256, 512]
est = [infonce_gauss(rho, K) for K in Ks]
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(Ks, est, "o-", color=PAL["azul"], label="log K − L_InfoNCE (crítico óptimo)")
ax.plot(Ks, np.log(Ks), "--", color=PAL["gris"], label="techo log K")
ax.axhline(I_real, color=PAL["naranja"], label=f"I(X;Y) real = {I_real:.2f} nats")
ax.set_xscale("log", base=2); ax.set(xlabel="K (nº de candidatos = tamaño del lote)", ylabel="nats", title="InfoNCE es una cota inferior saturada en log K")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print({K: round(e, 3) for K, e in zip(Ks, est)}, "| I real =", round(I_real, 3))

> 👀 **Qué debes observar.** La estimación sigue al techo $\log K$ (gris) mientras $\log K < I$ y se **satura** por debajo de la MI real (naranja, ≈ 1.96 nats) cuando $K$ es grande: nunca la supera. Con lotes de 8 la cota no puede pasar de $\log 8\approx2.08$ nats, aunque el modelo fuera perfecto. Esa es la razón matemática de que los embedders de producción se entrenen con lotes de miles.

## 3. Implementación desde cero: bi-encoder + InfoNCE
Tres piezas: (1) un tokenizer BPE entrenado en el corpus, (2) una torre `BiEncoder` (embeddings → 2 capas Transformer → media con máscara → L2), (3) `info_nce` con negativos en el lote, negativos duros opcionales y versión simétrica. Además, `alineacion` y `uniformidad` (sección 4).

In [ ]:
# ── Bi-encoder DESDE CERO: tokenizer BPE + Transformer encoder diminuto + mean pooling + L2 ──
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
from transformers import PreTrainedTokenizerFast


def entrena_tokenizer(textos, vocab_size=2000):
    tk = Tokenizer(models.BPE(unk_token="[UNK]"))
    tk.normalizer = normalizers.Sequence([normalizers.NFKC(), normalizers.Lowercase()])
    tk.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.Whitespace(), pre_tokenizers.Digits(individual_digits=True)])
    tk.train_from_iterator(textos, trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["[PAD]", "[UNK]"]))
    return PreTrainedTokenizerFast(tokenizer_object=tk, pad_token="[PAD]", unk_token="[UNK]")


class BiEncoder(nn.Module):
    """Una sola torre (pesos compartidos para consulta y documento): emb -> Transformer -> media con máscara -> L2."""

    def __init__(self, V, d=128, capas=2, cabezas=4, max_len=64, dropout=0.1):
        super().__init__()
        self.emb, self.pos = nn.Embedding(V, d, padding_idx=0), nn.Embedding(max_len, d)
        capa = nn.TransformerEncoderLayer(d, cabezas, 2 * d, dropout=dropout, batch_first=True, norm_first=True)
        self.enc, self.ln, self.max_len = nn.TransformerEncoder(capa, capas, enable_nested_tensor=False), nn.LayerNorm(d), max_len

    def forward(self, ids, mask):
        x = self.emb(ids) + self.pos(torch.arange(ids.shape[1], device=ids.device))[None]
        h = self.ln(self.enc(x, src_key_padding_mask=~mask.bool()))
        m = mask.unsqueeze(-1).float()
        return F.normalize((h * m).sum(1) / m.sum(1).clamp(min=1), dim=-1)


def tokeniza(tok, textos, max_len=64):
    b = tok(textos, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    return b["input_ids"].to(device), b["attention_mask"].to(device)


@torch.no_grad()
def codifica(model, tok, textos, bs=128):
    model.eval()
    return torch.cat([model(*tokeniza(tok, textos[i:i + bs])) for i in range(0, len(textos), bs)]).cpu()

In [ ]:
def info_nce(q, d, tau=0.05, d_extra=None, simetrica=False):
    """InfoNCE con negativos en el lote. q, d: (B, dim) normalizados; el positivo de q[i] es d[i].
    d_extra: (M, dim) negativos adicionales (hard negatives) que se añaden como columnas.
    simetrica=True suma también la dirección documento -> consulta."""
    D = d if d_extra is None else torch.cat([d, d_extra])
    logits = q @ D.T / tau                              # (B, B + M)
    y = torch.arange(q.shape[0], device=q.device)
    loss = F.cross_entropy(logits, y)
    if simetrica:
        loss = (loss + F.cross_entropy(d @ q.T / tau, y)) / 2
    return loss


def alineacion(x, y, alpha=2):
    """L_align = E ||f(x) - f(y)||^alpha sobre pares positivos (Wang & Isola, 2020). Menor = mejor."""
    return (x - y).norm(dim=1).pow(alpha).mean().item()


def uniformidad(x, t=2):
    """L_uniform = log E exp(-t ||f(x) - f(y)||^2) sobre pares aleatorios. Más negativo = más uniforme."""
    return torch.pdist(x, p=2).pow(2).mul(-t).exp().mean().log().item()

In [ ]:
# Test: info_nce == entropía cruzada calculada a mano sobre la matriz de similitudes
torch.manual_seed(SEED)
q_, d_ = F.normalize(torch.randn(4, 8), dim=-1), F.normalize(torch.randn(4, 8), dim=-1)
S_ = q_ @ d_.T / 0.05
manual = -(torch.diag(S_) - torch.logsumexp(S_, dim=1)).mean()
print(f"info_nce={info_nce(q_, d_, 0.05):.6f} | a mano={manual:.6f}")
h_ = F.normalize(torch.randn(3, 8), dim=-1)
print("con 3 negativos duros, la matriz pasa de 4×4 a", tuple((q_ @ torch.cat([d_, h_]).T).shape))

In [ ]:
def entrena_biencoder(pares, tok, pasos=300, B=32, tau=0.05, lr=2e-3, duros=None, sin_duplicados=True,
                      solo_positivos=False, simetrica=False, seed=SEED, grupo=None, log_cada=50, eval_fn=None, textos=None):
    """pares: lista de (texto_consulta, id_doc_positivo). duros: {índice_par: [ids de docs negativos]}.
    grupo: función par -> clave; sin_duplicados evita dos pares con la misma clave en un lote (falsos negativos).
    textos: dict id_doc -> texto (por defecto, el corpus Políglota)."""
    textos = textos or txt_of
    torch.manual_seed(seed); rng = random.Random(seed)
    model = BiEncoder(len(tok)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    grupo = grupo or (lambda p: p[1])
    hist = []
    for paso in range(pasos):
        orden = rng.sample(range(len(pares)), len(pares))
        lote, vistos = [], set()
        for i in orden:
            g = grupo(pares[i])
            if sin_duplicados and g in vistos:
                continue
            vistos.add(g); lote.append(i)
            if len(lote) == B:
                break
        q = model(*tokeniza(tok, [pares[i][0] for i in lote]))
        d = model(*tokeniza(tok, [textos[pares[i][1]] for i in lote]))
        extra = None
        if duros is not None:
            neg = [rng.choice(duros[i]) for i in lote if duros.get(i)]
            extra = model(*tokeniza(tok, [textos[n] for n in neg])) if neg else None
        if solo_positivos:
            loss = (q - d).pow(2).sum(1).mean()                 # sin negativos: ¡invita al colapso!
        else:
            loss = info_nce(q, d, tau, extra, simetrica)
        opt.zero_grad(); loss.backward(); opt.step()
        if paso % log_cada == 0 or paso == pasos - 1:
            reg = dict(paso=paso, loss=loss.item(), align=alineacion(q.detach(), d.detach()), unif=uniformidad(d.detach()))
            if eval_fn:
                reg["ndcg"] = eval_fn(model)
            hist.append(reg)
    return model.eval(), hist

> ### 🎬 GEMINI-VIDEO · ID: `M11-VID-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **La matriz B×B: cómo InfoNCE empuja positivos y negativos**
>
> **Qué muestra:** Anima un lote de 4 pares: matriz de similitudes, softmax por filas, efecto de la temperatura y de añadir un negativo duro como columna extra.
> **Formato:** MP4 16:9, 1080p | Duración: 40-45 s | **Texto alternativo:** Animación de una matriz 4×4 de similitudes, su softmax por filas con τ=0.05 y τ=1, el empuje sobre negativos difíciles y la adición de un negativo duro.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Video técnico 16:9, 1080p, fondo blanco, estilo de animación de diagramas (sin personajes, sin voz en off, sin música estridente; ritmo calmado). Paleta: verde #2CA02C (positivos/diagonal), gris #6B7280 (negativos), naranja #F28E2B (negativo duro), azul #1F6FEB (texto y ejes). Tipografía sans-serif grande.
> 0–5 s: aparece el título «InfoNCE con negativos en el lote» y, debajo, cuatro filas «q1 q2 q3 q4» y cuatro columnas «d1 d2 d3 d4».
> 5–12 s: se rellena una matriz 4×4 de similitudes coseno con estos valores exactos (fila por fila): q1: 0.82, 0.31, 0.12, 0.78 · q2: 0.28, 0.77, 0.40, 0.05 · q3: 0.10, 0.35, 0.69, 0.66 · q4: 0.47, 0.02, 0.18, 0.80. La diagonal se colorea en verde; las celdas (q1,d4)=0.78 y (q3,d4)=0.66 parpadean en naranja con el rótulo «negativos difíciles».
> 12–20 s: texto en pantalla «softmax por filas con τ = 0.05». Cada fila se transforma en probabilidades exactas: q1: 0.69, 0.00, 0.00, 0.31 · q2: 0.00, 1.00, 0.00, 0.00 · q3: 0.00, 0.00, 0.65, 0.35 · q4: 0.00, 0.00, 0.00, 1.00. Junto a cada fila aparece su pérdida −log p(diagonal): 0.37, 0.00, 0.44, 0.00; abajo «pérdida media = 0.20».
> 20–27 s: texto «la misma matriz con τ = 1.0». Las filas se vuelven casi uniformes: q1: 0.33, 0.20, 0.16, 0.31 · q2: 0.22, 0.36, 0.25, 0.17 · q3: 0.17, 0.22, 0.31, 0.30 · q4: 0.26, 0.17, 0.20, 0.37; «pérdida media = 1.08». Rótulo: «τ alta: todos los negativos pesan casi igual».
> 27–35 s: flechas animadas sobre una esfera pequeña a la derecha: el punto q1 se acerca a d1 (flecha verde) y se aleja sobre todo de d4 (flecha naranja más gruesa) y poco de d2, d3 (flechas grises finas). Rótulo: «el gradiente se concentra en los negativos difíciles (τ baja)».
> 35–42 s: se añade a la derecha de la matriz una quinta columna naranja rotulada «h1 (negativo duro minado con BM25)», la matriz pasa a 4×5; rótulo final: «negativos duros = columnas extra del denominador».
> Evita: texto ilegible, números distintos de los indicados, transiciones rápidas, elementos decorativos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M11-VID-01.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M11-VID-01.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
> S (cos)   d1   d2   d3   d4        softmax τ=0.05            τ=1.0
> q1      [0.82 0.31 0.12 0.78]  →  [0.69 0.00 0.00 0.31]   [0.33 0.20 0.16 0.31]
> q2      [0.28 0.77 0.40 0.05]  →  [0.00 1.00 0.00 0.00]   [0.22 0.36 0.25 0.17]
> q3      [0.10 0.35 0.69 0.66]  →  [0.00 0.00 0.65 0.35]   [0.17 0.22 0.31 0.30]
> q4      [0.47 0.02 0.18 0.80]  →  [0.00 0.00 0.00 1.00]   [0.26 0.17 0.20 0.37]
> pérdida media: 0.20 (τ=0.05) · 1.08 (τ=1)   + columna h1 = negativo duro
> ```

> 👀 **Qué debes observar.** Con $\tau=0.05$ las filas 2 y 4 ya no aportan pérdida (sus negativos son fáciles) y todo el gradiente sale de las filas 1 y 3, por culpa de un único negativo difícil cada una. Con $\tau=1$ la pérdida es alta en todas las filas pero el gradiente apenas distingue difíciles de fáciles. Los números del video salen de aplicar la fórmula de la sección 2 a esa matriz.

## 4. Laboratorio controlado: temperatura, colapso, alineación y uniformidad

Para ver efectos limpios usamos un **mundo de juguete** con verdad conocida: cada «concepto» es (acción, objeto) — p. ej. *(ordenar, lista)* — descrito en ES, EN y código con **sinónimos** distintos en consultas y documentos, y todos los documentos del concepto son relevantes (también los del otro idioma). Se entrena con 48 conceptos y se evalúa en **12 conceptos nunca vistos** (combinaciones nuevas de palabras conocidas). BM25 aquí falla por la brecha léxica y el cruce de idioma; un embedder puede aprenderlos.

**Alineación y uniformidad** (Wang & Isola, 2020), sobre la hiperesfera:
$$\mathcal L_{\text{align}}=\mathbb E_{(x,y)\sim p_{\text{pos}}}\|f(x)-f(y)\|^{2},\qquad \mathcal L_{\text{unif}}=\log\mathbb E_{x,y\sim p_{\text{data}}}e^{-2\|f(x)-f(y)\|^2}.$$
Alineación baja = los positivos están cerca; uniformidad **muy negativa** = los puntos cubren la esfera. Un **colapso** (todo al mismo punto) tiene alineación perfecta (0) y uniformidad ≈ 0: el peor caso. InfoNCE optimiza asintóticamente ambas a la vez.

In [ ]:
# ── Léxico del corpus SINTÉTICO (fallback): un "concepto" = (acción, objeto), con dos sinónimos por idioma ──
import random
import re
from collections import Counter

ACCIONES = {  # id: (sinónimos ES, sinónimos EN, verbo en código)
    "sort": (["ordenar", "clasificar"], ["sort", "order"], "sort"),
    "filter": (["filtrar", "depurar"], ["filter", "screen"], "filter"),
    "merge": (["combinar", "fusionar"], ["merge", "combine"], "merge"),
    "split": (["dividir", "separar"], ["split", "divide"], "split"),
    "delete": (["eliminar", "borrar"], ["delete", "remove"], "delete"),
    "copy": (["copiar", "duplicar"], ["copy", "duplicate"], "copy"),
    "encrypt": (["cifrar", "encriptar"], ["encrypt", "scramble"], "encrypt"),
    "compress": (["comprimir", "empaquetar"], ["compress", "pack"], "compress"),
    "validate": (["validar", "verificar"], ["validate", "verify"], "validate"),
    "parse": (["analizar", "interpretar"], ["parse", "interpret"], "parse"),
    "download": (["descargar", "bajar"], ["download", "fetch"], "download"),
    "upload": (["subir", "cargar"], ["upload", "push"], "upload"),
}
OBJETOS = {
    "list": (["lista", "arreglo"], ["list", "array"], "list"),
    "file": (["archivo", "fichero"], ["file", "document"], "file"),
    "string": (["cadena", "texto"], ["string", "text"], "string"),
    "table": (["tabla", "planilla"], ["table", "spreadsheet"], "table"),
    "image": (["imagen", "foto"], ["image", "picture"], "image"),
    "message": (["mensaje", "aviso"], ["message", "notification"], "message"),
    "user": (["usuario", "cuenta"], ["user", "account"], "user"),
    "log": (["registro", "bitácora"], ["log", "journal"], "log"),
    "password": (["contraseña", "clave"], ["password", "passphrase"], "password"),
    "link": (["enlace", "vínculo"], ["link", "address"], "link"),
    "matrix": (["matriz", "cuadrícula"], ["matrix", "grid"], "matrix"),
    "date": (["fecha", "calendario"], ["date", "calendar"], "date"),
}
RELLENO = {
    "es": ["Funciona en Linux y en macOS.", "Requiere Python 3.10 o superior.", "Revisa la configuración del servidor antes de continuar.",
           "El resultado se guarda en el directorio de trabajo.", "Consulta los registros si aparece un error.",
           "Esta operación es idempotente y segura en producción.", "Puedes automatizarlo con un script de shell.",
           "Se recomienda hacer una copia de seguridad previa.", "El tiempo de ejecución crece con el tamaño de los datos."],
    "en": ["It works on Linux and macOS.", "Requires Python 3.10 or later.", "Check the server configuration before continuing.",
           "The result is stored in the working directory.", "Look at the logs if an error shows up.",
           "This operation is idempotent and safe in production.", "You can automate it with a shell script.",
           "Making a backup first is recommended.", "Running time grows with the size of the data."],
}
PLANTILLAS = {
    "es": ["Cómo {a} {o} paso a paso.", "Guía rápida para {a} {o} correctamente.", "Tutorial: {a} {o} en tu proyecto."],
    "en": ["How to {a} a {o} step by step.", "Quick guide to {a} the {o} correctly.", "Tutorial: {a} the {o} in your project."],
}

In [ ]:
def _frase(rng, lang, a, o, estilo):
    idx = 0 if lang == "es" else 1
    return PLANTILLAS[lang][estilo % 3].format(a=rng.choice(ACCIONES[a][idx]), o=rng.choice(OBJETOS[o][idx]))


def _doc_codigo(rng, a, o):
    fa, fo = ACCIONES[a][2], OBJETOS[o][2]
    va, vo = rng.choice(ACCIONES[a][1]), rng.choice(OBJETOS[o][1])
    return (f"def {fa}_{fo}({fo}, strict=False):\n    \"\"\"{va.capitalize()} the {vo} and return the result.\"\"\"\n"
            f"    result = helper_{fa}({fo})\n    return result")


def genera_mundo_juguete(fast_dev: bool = True, seed: int = 42) -> dict:
    """Corpus sintético con el MISMO esquema que el real. Concepto = (acción, objeto); relevantes = TODOS los docs
    del concepto (ES, EN y código). Las queries usan sinónimos distintos: hay brecha léxica y cruce de idioma.
    Extras no contractuales: docs[i]['concept'], queries[i]['concept'] y queries[i]['split'] ('train'|'test', por concepto)."""
    rng = random.Random(seed)
    acts, objs = sorted(ACCIONES), sorted(OBJETOS)
    conceptos = [(a, o) for a in acts for o in objs]
    rng.shuffle(conceptos)
    conceptos = conceptos[:60] if fast_dev else conceptos
    docs, queries, qrels = [], [], {}
    for ci, (a, o) in enumerate(conceptos):
        cid, rel = f"c{ci:03d}", {}
        for lang, n in (("es", 3), ("en", 3), ("code", 2)):
            for k in range(n):
                did = f"{cid}-{lang}{k}"
                if lang == "code":
                    text = _doc_codigo(rng, a, o)
                else:
                    a2, o2 = rng.choice(acts), rng.choice(objs)
                    otra = _frase(rng, lang, a2, o2, k + 1) if (a2, o2) != (a, o) else ""
                    extra = (("No confundir con: " if lang == "es" else "Do not confuse with: ") + otra) if otra else ""
                    text = " ".join([_frase(rng, lang, a, o, k), rng.choice(RELLENO[lang]), rng.choice(RELLENO[lang]), extra]).strip()
                docs.append({"id": did, "lang": lang, "text": text, "concept": cid})
                rel[did] = 1
        for k, lang in enumerate(("es", "es", "en", "en")):
            qid = f"q{ci:03d}-{k}"
            idx = 0 if lang == "es" else 1
            text = f"{rng.choice(ACCIONES[a][idx])} {rng.choice(OBJETOS[o][idx])}"
            queries.append({"qid": qid, "lang": lang, "text": text, "concept": cid, "split": "test" if ci % 5 == 0 else "train"})
            qrels[qid] = dict(rel)
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "toy"}

In [ ]:
# ── Laboratorio controlado: "mundo de juguete" con verdad conocida (concepto = acción × objeto, ES/EN/código) ──
toy = genera_mundo_juguete(fast_dev=True)
toy_txt = {d["id"]: d["text"] for d in toy["docs"]}
toy_ids = list(toy_txt)
concepto = {d["id"]: d["concept"] for d in toy["docs"]}
por_concepto = defaultdict(list)
for d in toy["docs"]:
    por_concepto[d["concept"]].append(d["id"])
toy_tok = entrena_tokenizer(list(toy_txt.values()) + [q["text"] for q in toy["queries"]], vocab_size=800)
toy_q_train = [q for q in toy["queries"] if q["split"] == "train"]
toy_q_test = [q for q in toy["queries"] if q["split"] == "test"]          # conceptos NO vistos al entrenar
toy_pares = [(q["text"], did) for q in toy_q_train for did in por_concepto[q["concept"]]]


def eval_toy(model, qs=None):
    qs = qs or toy_q_test
    D = codifica(model, toy_tok, [toy_txt[i] for i in toy_ids]); Q = codifica(model, toy_tok, [q["text"] for q in qs])
    run = {q["qid"]: rank_from_scores((Q @ D.T)[k].numpy(), toy_ids, 100) for k, q in enumerate(qs)}
    return evaluate(run, {q["qid"]: toy["qrels"][q["qid"]] for q in qs})["ndcg"]


print(f"docs={len(toy_ids)} | conceptos={len(por_concepto)} | pares de entrenamiento={len(toy_pares)} | queries test={len(toy_q_test)}")
print("ejemplo:", toy_q_train[0]["text"], "→", toy_txt[por_concepto[toy_q_train[0]["concept"]][0]][:80])

In [ ]:
PASOS_LAB = 200 if FAST_DEV_RUN else 1000
grupo_toy = lambda p: concepto[p[1]]                     # evita dos pares del mismo concepto en un lote
lab = {}
t0 = time.time()
for tau in (0.02, 0.05, 0.2, 1.0):
    m, h = entrena_biencoder(toy_pares, toy_tok, pasos=PASOS_LAB, B=32, tau=tau, grupo=grupo_toy, textos=toy_txt)
    lab[f"τ={tau}"] = dict(ndcg=eval_toy(m), align=h[-1]["align"], unif=h[-1]["unif"])
m, h = entrena_biencoder(toy_pares, toy_tok, pasos=PASOS_LAB, B=32, solo_positivos=True, grupo=grupo_toy, textos=toy_txt)
lab["sin negativos (colapso)"] = dict(ndcg=eval_toy(m), align=h[-1]["align"], unif=h[-1]["unif"])
m0 = BiEncoder(len(toy_tok)).to(device)
lab["sin entrenar"] = dict(ndcg=eval_toy(m0), align=float("nan"), unif=float("nan"))
for k, v in lab.items():
    print(f"{k:24s} nDCG@10(test)={v['ndcg']:.3f} | align={v['align']:.3f} | unif={v['unif']:.3f}")
print(f"({time.time() - t0:.0f}s)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
nombres = list(lab)
cols = [PAL["azul"]] * 4 + [PAL["naranja"], PAL["gris"]]
ax[0].bar(range(len(nombres)), [lab[n]["ndcg"] for n in nombres], color=cols)
ax[0].set_xticks(range(len(nombres)), nombres, rotation=25, ha="right"); ax[0].set(ylabel="nDCG@10 en conceptos no vistos", title="Temperatura y colapso")
for n, c in zip(nombres[:5], cols):
    ax[1].scatter(lab[n]["unif"], lab[n]["align"], s=70, color=c)
    ax[1].annotate(n, (lab[n]["unif"], lab[n]["align"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax[1].set(xlabel="L_uniform (← más uniforme)", ylabel="L_align (↓ positivos más cerca)", title="Plano alineación–uniformidad (Wang & Isola)")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** (1) Hay un $\tau$ intermedio que gana (en la ejecución de referencia, $\tau=0.2$ ≈ 1.00 de nDCG@10 frente a 0.49 con $\tau=1$); $\tau=1$ no separa bien porque todos los negativos pesan igual. (2) **Sin negativos** el modelo colapsa: alineación ≈ 0 (¡perfecta!) pero uniformidad ≈ 0 y nDCG cercano al de un modelo sin entrenar. Por eso no basta con «acercar positivos»: los negativos (o un regularizador de uniformidad) son los que impiden el colapso.
> (3) En el plano, los buenos modelos están **abajo a la izquierda**: positivos cerca *y* puntos repartidos. El valor óptimo de $\tau$ depende de datos, tamaño de lote y modelo: SimCSE usó 0.05 y E5 0.01; trátalo como hiperparámetro.

> ### 🎨 GEMINI-IMAGEN · ID: `M11-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Alineación y uniformidad en la hiperesfera**
>
> **Qué muestra:** Tres esferas que ilustran colapso, alineación sin uniformidad y el caso ideal, conectando con L_align y L_uniform.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Tres esferas: colapso en un punto, grupos alineados pero apiñados, y grupos repartidos uniformemente.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9, tres círculos grandes (esferas vistas en 2D) en fila con título encima de cada uno. Paleta: cada color es una clase de pares positivos: azul #1F6FEB, naranja #F28E2B, verde #2CA02C; gris #6B7280 para el contorno y texto.
> Esfera 1, título «Colapso»: todos los puntos (de los tres colores) amontonados en un único punto del borde. Debajo: «L_align ≈ 0 · L_uniform ≈ 0 → inútil».
> Esfera 2, título «Alineado pero no uniforme»: tres grupos compactos (azul, naranja, verde) todos apiñados en el mismo cuarto de la esfera, muy cerca unos de otros. Debajo: «positivos cerca, pero las clases se confunden».
> Esfera 3, título «Ideal (InfoNCE)»: tres grupos compactos repartidos alrededor de la esfera, separados ~120°. Debajo: «L_align bajo · L_uniform muy negativo».
> Pie común con las fórmulas: «L_align = E‖f(x) − f(y)‖² (pares positivos)» y «L_uniform = log E exp(−2‖f(x) − f(y)‖²) (pares aleatorios)».
> Evita: texto ilegible, perspectiva 3D confusa, números inventados, decoración.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M11-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Tres esferas: colapso en un punto, grupos alineados pero apiñados, y grupos repartidos uniformemente.](assets/M11-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> (1) Colapso          (2) Alineado, no uniforme     (3) Ideal
>    ( ●●● )              ( ●● ▲▲ ■■   )              ( ●●      ▲▲ )
> align≈0, unif≈0        positivos cerca, clases      (     ■■      )
>                        apiñadas                     align bajo, unif muy negativo
> ```

> 👀 **Qué debes observar.** El caso 2 es el que produce un $\tau$ demasiado alto o pocos negativos: la alineación por sí sola no garantiza que clases distintas queden lejos. Relaciónalo con la anisotropía del módulo 10: un espacio «apiñado» es anisótropo.

## 5. Negativos duros y falsos negativos
Los negativos del lote son **aleatorios**: la mayoría son fáciles y dejan de dar gradiente pronto (lo viste en el video). Los **negativos duros** son documentos parecidos a la consulta pero no relevantes; se minan con:
- **BM25** (DPR, Karpukhin et al., 2020: su mejor modelo usaba los positivos del lote + **un** negativo de BM25 por pregunta, con lotes de 128).
- **El propio modelo denso**, refrescando un índice ANN durante el entrenamiento (ANCE, Xiong et al., 2020).
- **Un cross-encoder** para filtrar falsos negativos (RocketQA, Qu et al., 2021) o **destilar** (TAS-B, Hofstätter et al., 2021).

**El peligro: falsos negativos.** El top de BM25 contiene a menudo documentos **relevantes no etiquetados**; tratarlos como negativos enseña al modelo lo contrario de lo que queremos. NV-Retriever (Moreira et al., 2024) propone filtros «conscientes del positivo»; en su configuración principal (*TopK-PercPos*) descartan los candidatos cuyo score supera el **95 % del score del positivo**.

In [ ]:
def mina_duros(pares, bm, ids, textos, k=30, n=10, perc=0.95, mismo_grupo=None):
    """Negativos duros con BM25. perc: filtro TopK-PercPos (score < perc · score del positivo; None = sin filtro).
    mismo_grupo(did_pos, did_cand) -> True si el candidato es un falso negativo conocido (solo en el mundo de juguete)."""
    pos_idx = {d: i for i, d in enumerate(ids)}
    duros = {}
    for k_, (qt, did) in enumerate(pares):
        sc = bm.scores(tokenize_simple(qt))
        s_pos = sc[pos_idx[did]]
        cand = [ids[j] for j in np.argsort(-sc)[:k] if ids[j] != did]
        if perc is not None:
            cand = [c for c in cand if sc[pos_idx[c]] < perc * s_pos]
        if mismo_grupo is not None:
            cand = [c for c in cand if not mismo_grupo(did, c)]
        duros[k_] = cand[:n]
    return duros


toy_bm = BM25().fit([tokenize_simple(toy_txt[i]) for i in toy_ids], toy_ids)
duros_crudos = mina_duros(toy_pares, toy_bm, toy_ids, toy_txt, perc=None)
duros_limpios = mina_duros(toy_pares, toy_bm, toy_ids, toy_txt, perc=None, mismo_grupo=lambda a, b: concepto[a] == concepto[b])
fn = np.mean([np.mean([concepto[c] == concepto[toy_pares[k][1]] for c in v]) for k, v in duros_crudos.items() if v])
print(f"fracción de FALSOS negativos en el top-10 de BM25 sin filtrar: {fn:.0%}")

In [ ]:
neg = {}
for nombre, du in (("solo en el lote", None), ("+ duros limpios", duros_limpios), ("+ duros sin filtrar", duros_crudos)):
    m, _ = entrena_biencoder(toy_pares, toy_tok, pasos=PASOS_LAB, B=16, tau=0.05, duros=du, grupo=grupo_toy, textos=toy_txt)
    neg[nombre] = eval_toy(m)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar(list(neg), list(neg.values()), color=[PAL["gris"], PAL["verde"], PAL["naranja"]])
for i, v in enumerate(neg.values()):
    ax.text(i, v, f"{v:.3f}", ha="center", va="bottom")
ax.set(ylabel="nDCG@10 (conceptos no vistos)", title="Negativos duros: limpios ayudan, falsos negativos dañan (B=16, τ=0.05)")
plt.tight_layout(); plt.show()
print({k: round(v, 3) for k, v in neg.items()})

> 👀 **Qué debes observar.** En el prototipo de referencia (mismo código, 300 pasos): solo lote 0.88 · con duros limpios 0.92 · con duros **sin filtrar** 0.84. Con `FAST_DEV_RUN` (200 pasos) la ejecución de referencia dio 0.766 · 0.728 · 0.754: con tan pocos pasos el modelo aún no domina los negativos fáciles y las diferencias son del orden del ruido (el orden puede invertirse); con `FAST_DEV_RUN=False` (1000 pasos) deberías ver el efecto. El top de BM25 sin filtrar está lleno de documentos del **mismo concepto** (falsos negativos, mira el porcentaje impreso): con entrenamiento suficiente, usarlos como negativos es peor que no usar negativos duros. En datos reales no conoces el concepto: de ahí los filtros por score (TopK-PercPos) o por cross-encoder.

> ### 🎨 GEMINI-IMAGEN · ID: `M11-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **Minado de negativos duros con filtro de falsos negativos**
>
> **Qué muestra:** Pipeline de minado: candidatos BM25/denso, filtro consciente del positivo y negativos que entran en la pérdida.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Flujo: consulta → recuperador con 6 candidatos y scores → filtro 95 % del positivo que descarta c1 → negativos duros c2-c4 a la pérdida.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9, flujo de izquierda a derecha. Paleta: azul #1F6FEB (consulta), verde #2CA02C (positivo), naranja #F28E2B (negativos duros válidos), rojo apagado #C0392B (falsos negativos descartados), gris #6B7280 (texto secundario).
> 1) Caja azul «consulta: configurar el balanceador de carga».
> 2) Caja «Recuperador (BM25 o modelo denso)» que produce una lista vertical de 6 candidatos con scores exactos: «d+ (positivo) 12.0», «c1 11.8», «c2 10.9», «c3 9.1», «c4 7.5», «c5 6.2».
> 3) Caja «Filtro TopK-PercPos: score < 0.95 × 12.0 = 11.4». c1 (11.8) aparece en rojo tachado con la etiqueta «posible falso negativo»; c2 (10.9), c3, c4 en naranja con «negativo duro»; c5 en gris «fácil».
> 4) Caja final «InfoNCE: positivo + negativos del lote + c2, c3, c4 como columnas extra».
> Nota al pie: «Alternativas: filtrar con un cross-encoder (RocketQA) o refrescar el índice durante el entrenamiento (ANCE)».
> Evita: texto ilegible, flechas ambiguas, otros números, decoración.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M11-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Flujo: consulta → recuperador con 6 candidatos y scores → filtro 95 % del positivo que descarta c1 → negativos duros c2-c4 a la pérdida.](assets/M11-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> consulta → [BM25 / denso] → d+ 12.0 | c1 11.8 | c2 10.9 | c3 9.1 | c4 7.5 | c5 6.2
>          → filtro: score < 0.95·12.0 = 11.4 → c1 ✗ (posible falso negativo)
>          → negativos duros: c2, c3, c4  → InfoNCE (columnas extra)
> ```

> 👀 **Qué debes observar.** El candidato más parecido al positivo es el **más sospechoso**, no el mejor negativo. El umbral relativo (95 % del positivo) se adapta a cada consulta, a diferencia de un umbral absoluto.

## 6. SimCSE, DPR y la receta «débil → supervisado»
- **SimCSE no supervisado** (Gao, Yao & Chen, 2021): el positivo de una frase es **ella misma** pasada dos veces por el encoder con *dropout* distinto; los negativos, las demás frases del lote ($\tau=0.05$). Sin etiquetas. **SimCSE supervisado**: pares de inferencia natural (implicación = positivo, contradicción = negativo duro).
- **DPR** (Karpukhin et al., 2020): dos encoders BERT (pregunta y pasaje), producto escalar sobre `[CLS]`, negativos del lote + BM25.
- **Pre-entrenamiento débilmente supervisado → ajuste fino.** La receta de los embedders modernos (E5, GTE, BGE): primero cientos de millones de **pares ruidosos** extraídos de la web (título–cuerpo, pregunta–respuesta, post–comentario); E5 construyó **CCPairs, ≈ 270M pares**, con $\tau=0.01$ y prefijos `query:`/`passage:` (Wang et al., 2022). Después, un ajuste fino con datos etiquetados y negativos duros. GTE (Li et al., 2023) y BGE / C-Pack (Xiao et al., 2023) describen variantes multi-etapa del mismo esquema.

Probamos SimCSE no supervisado en el mundo de juguete (pares $(x, x)$ con dropout):

In [ ]:
textos_s = list(toy_txt.values()) + [q["text"] for q in toy_q_train]
ids_s = {f"s{i}": x for i, x in enumerate(textos_s)}
pares_simcse = [(x, f"s{i}") for i, x in enumerate(textos_s)]            # positivo = la misma frase (dropout distinto)
m_simcse, _ = entrena_biencoder(pares_simcse, toy_tok, pasos=PASOS_LAB, B=32, tau=0.05, textos=ids_s)
print(f"nDCG@10 test · sin entrenar={lab['sin entrenar']['ndcg']:.3f} | SimCSE no supervisado={eval_toy(m_simcse):.3f} | supervisado τ=0.05={lab['τ=0.05']['ndcg']:.3f}")

> 👀 **Qué debes observar.** SimCSE sin etiquetas mejora claramente al modelo sin entrenar (aprende a ignorar ruido y a agrupar textos parecidos), pero queda muy lejos del supervisado: **no puede** aprender que «bajar» = «descargar» = «download» sin señal de pares. Por eso los embedders de producción usan pares débiles a escala web, no solo auto-supervisión.

## 7. Lotes grandes y Gradient Cache
La cota $\log K$ y la práctica dicen lo mismo: **más negativos en el lote ayudan**. Pero la memoria de activaciones crece con $B$. **Gradient Cache** (Gao et al., 2021) desacopla ambas cosas: (1) forward sin grafo de todo el lote para obtener las representaciones; (2) pérdida y gradiente **respecto a las representaciones** (barato: solo vectores); (3) re-forward por trozos con grafo y backward de $\sum_i \mathbf z_i\cdot\partial\mathcal L/\partial\mathbf z_i$. El gradiente es **exacto**; el coste es ≈ un forward extra. En `sentence-transformers` es `CachedMultipleNegativesRankingLoss`.

In [ ]:
B_list = [4, 8, 16, 32]
por_B = {}
for B_ in B_list:
    m, _ = entrena_biencoder(toy_pares, toy_tok, pasos=PASOS_LAB, B=B_, tau=0.05, grupo=grupo_toy, textos=toy_txt)
    por_B[B_] = eval_toy(m)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(B_list, list(por_B.values()), "o-", color=PAL["azul"])
ax.set_xscale("log", base=2); ax.set(xlabel="tamaño de lote B (mismo nº de pasos)", ylabel="nDCG@10 (conceptos no vistos)", title="Más negativos en el lote")
plt.tight_layout(); plt.show()
print(por_B)

> 👀 **Qué debes observar.** Con el mismo número de pasos, lotes mayores dan mejor nDCG (más negativos por paso *y* más ejemplos vistos: ambos efectos se mezclan). En el mundo de juguete el lote está limitado por el nº de conceptos de entrenamiento (48) si exigimos que no se repitan: con datos reales el límite es la **memoria**, y ahí entra Gradient Cache.

In [ ]:
def paso_gradcache(model, tok, q_txt, d_txt, tau=0.05, trozo=8):
    """Un paso de Gradient Cache (Gao et al., 2021): gradiente EXACTO de InfoNCE con lote B usando memoria de 'trozo'.
    1) forward SIN grafo de todo el lote -> representaciones; 2) pérdida y gradiente respecto a las representaciones;
    3) re-forward por trozos CON grafo y backward del producto (representación · gradiente cacheado)."""
    with torch.no_grad():
        zq = torch.cat([model(*tokeniza(tok, q_txt[i:i + trozo])) for i in range(0, len(q_txt), trozo)])
        zd = torch.cat([model(*tokeniza(tok, d_txt[i:i + trozo])) for i in range(0, len(d_txt), trozo)])
    zq.requires_grad_(True); zd.requires_grad_(True)
    loss = info_nce(zq, zd, tau)
    loss.backward()                                       # solo llega a zq, zd (no hay grafo del modelo)
    for textos_, g in ((q_txt, zq.grad), (d_txt, zd.grad)):
        for i in range(0, len(textos_), trozo):
            r = model(*tokeniza(tok, textos_[i:i + trozo]))
            (r * g[i:i + trozo]).sum().backward()         # regla de la cadena: dL/dθ = Σ (dL/dz)·(dz/dθ)
    return loss.detach()

In [ ]:
# Comprobación: el gradiente de GradCache coincide con el del lote completo (modo eval: sin dropout)
torch.manual_seed(SEED)
m_gc = BiEncoder(len(toy_tok)).to(device).eval()
lote = random.Random(SEED).sample(toy_pares, 32)
qs_, ds_ = [p[0] for p in lote], [toy_txt[p[1]] for p in lote]
m_gc.zero_grad()
info_nce(m_gc(*tokeniza(toy_tok, qs_)), m_gc(*tokeniza(toy_tok, ds_))).backward()
g_full = [p.grad.clone() for p in m_gc.parameters() if p.grad is not None]
m_gc.zero_grad()
paso_gradcache(m_gc, toy_tok, qs_, ds_, trozo=8)
g_gc = [p.grad.clone() for p in m_gc.parameters() if p.grad is not None]
dif = max((a - b).abs().max().item() for a, b in zip(g_full, g_gc))
print(f"max |grad lote completo − grad GradCache| = {dif:.2e}  (trozos de 8 para un lote de 32)")

> 👀 **Qué debes observar.** Diferencia del orden de $10^{-7}$: error de redondeo en float32. Gradient Cache no es una aproximación. Con *dropout* activo hay que reproducir el mismo estado aleatorio en los dos forwards (la implementación original guarda y restaura el RNG); aquí lo evitamos con `eval()`.

## 8. Políglota: tu primer embedder propio (desde cero)
Pares de entrenamiento:
- **Débiles** (sin etiquetas, como CCPairs en miniatura): primera frase del documento → documento; primera línea (firma) de cada función → función.
- **Etiquetados**: consultas de la partición de entrenamiento → su documento relevante.
Evaluamos en las consultas de **test** contra BM25 (módulo 00) y contra la misma torre sin entrenar.

In [ ]:
def pares_debiles(docs):
    out = []
    for d in docs:
        if d["lang"] == "code":
            out.append((d["text"].split("\n")[0], d["id"]))
        else:
            fr = re.split(r"(?<=[.!?])\s+", d["text"].strip())
            if len(fr) >= 3:
                out.append((fr[0], d["id"]))
    return out


pares_poliglota = pares_debiles(docs) + [(q["text"], did) for q in q_train for did in qrels[q["qid"]]] * 5
pol_tok = entrena_tokenizer(doc_txt + [q["text"] for q in queries], vocab_size=2000)
bm25_pol = BM25().fit([tokenize_simple(t) for t in doc_txt], doc_ids)


def ndcg_pol(model, qs=None):
    qs = qs or q_test
    D, Q = codifica(model, pol_tok, doc_txt), codifica(model, pol_tok, [q["text"] for q in qs])
    run = {q["qid"]: rank_from_scores((Q @ D.T)[k].numpy(), doc_ids, 100) for k, q in enumerate(qs)}
    return evaluate(run, {q["qid"]: qrels[q["qid"]] for q in qs})["ndcg"]


ndcg_bm25 = evaluate({q["qid"]: bm25_pol.search(tokenize_simple(q["text"]), 100) for q in q_test}, {q["qid"]: qrels[q["qid"]] for q in q_test})["ndcg"]
PASOS_POL = 300 if FAST_DEV_RUN else 3000
m_pol, hist_pol = entrena_biencoder(pares_poliglota, pol_tok, pasos=PASOS_POL, B=32, tau=0.05, log_cada=50, eval_fn=ndcg_pol)
print(f"pares={len(pares_poliglota)} | BM25={ndcg_bm25:.3f} | sin entrenar={ndcg_pol(BiEncoder(len(pol_tok)).to(device)):.3f} | entrenado={ndcg_pol(m_pol):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
pasos = [h["paso"] for h in hist_pol]
ax[0].plot(pasos, [h["loss"] for h in hist_pol], "o-", color=PAL["azul"]); ax[0].set(xlabel="paso", ylabel="InfoNCE", title="Pérdida de entrenamiento")
ax[1].plot(pasos, [h["ndcg"] for h in hist_pol], "o-", color=PAL["verde"], label="bi-encoder propio")
ax[1].axhline(ndcg_bm25, ls="--", color=PAL["naranja"], label=f"BM25 = {ndcg_bm25:.3f}")
ax[1].set(xlabel="paso", ylabel="nDCG@10 (test)", title="Políglota: calidad durante el entrenamiento"); ax[1].legend()
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** La pérdida baja de forma continua, pero el nDCG de test **no** tiene por qué hacerlo: con pocas consultas etiquetadas el modelo memoriza pronto. Y en este benchmark de ítem conocido, BM25 sigue muy por encima de una torre de 2 capas entrenada minutos en CPU: lo correcto es **ajustar un modelo preentrenado** (siguiente celda) y combinar denso + BM25 (módulo 13). Construir la torre desde cero sirve para entender cada pieza, no para ganar a BM25.

## 9. Con la librería de industria: `sentence-transformers`
`MultipleNegativesRankingLoss` (MNRL) es exactamente nuestro `info_nce` con negativos en el lote (y, si el dataset trae una tercera columna, negativos duros como columnas extra). Sus piezas habituales: `BatchSamplers.NO_DUPLICATES` (evita textos repetidos en un lote, que serían falsos negativos), `scale=20` (= $1/\tau$), y `CachedMultipleNegativesRankingLoss` para lotes grandes con Gradient Cache.

In [ ]:
# ── Con la librería de industria: sentence-transformers + MultipleNegativesRankingLoss ──
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.models import Pooling, Transformer
from sentence_transformers.training_args import BatchSamplers


def modelo_st():
    """Modelo real multilingüe si hay hub; si no, un mini-BERT local SIN preentrenar envuelto en sentence-transformers."""
    try:
        return SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device), "paraphrase-multilingual-MiniLM-L12-v2"
    except Exception as e:
        print(f"⚠️  sin hub ({type(e).__name__}): mini-BERT local aleatorio (fallback)")
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
    from transformers import BertConfig, BertModel, BertTokenizerFast
    os.makedirs("mini_st", exist_ok=True)
    # Vocabulario WordPiece REPRODUCIBLE: el WordPiece trainer de `tokenizers` rompe empates según un HashMap y cambia en
    # cada proceso; Unigram es determinista y lo convertimos a la convención WordPiece ("##" = continuación).
    uni = Tokenizer(models.Unigram()); uni.normalizer = normalizers.BertNormalizer(lowercase=True, strip_accents=False)
    uni.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.BertPreTokenizer(), pre_tokenizers.Metaspace()])
    uni.train_from_iterator(doc_txt, trainers.UnigramTrainer(vocab_size=2000, unk_token="[UNK]", special_tokens=["[UNK]"], show_progress=False))
    chars = sorted({c for t in doc_txt for c in uni.normalizer.normalize_str(t) if not c.isspace()})
    piezas = [p for p, _ in sorted(uni.get_vocab().items(), key=lambda x: x[1]) if p not in ("[UNK]", "▁")]
    vocab = list(dict.fromkeys(["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + chars + ["##" + c for c in chars]
                               + [p[1:] if p.startswith("▁") else "##" + p for p in piezas]))
    with open("mini_st/vocab.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(vocab) + "\n")
    BertTokenizerFast("mini_st/vocab.txt", do_lower_case=True, strip_accents=False).save_pretrained("mini_st")
    torch.manual_seed(SEED)
    BertModel(BertConfig(vocab_size=len(vocab), hidden_size=128, num_hidden_layers=2, num_attention_heads=4, intermediate_size=256,
                         max_position_embeddings=128)).save_pretrained("mini_st")
    tr = Transformer("mini_st", max_seq_length=64)
    return SentenceTransformer(modules=[tr, Pooling(tr.get_word_embedding_dimension(), "mean")], device=device), "mini-BERT local (fallback)"


def ndcg_st(model, qs):
    D = model.encode(doc_txt, convert_to_tensor=True, normalize_embeddings=True, batch_size=64)
    Q = model.encode([q["text"] for q in qs], convert_to_tensor=True, normalize_embeddings=True, batch_size=64)
    S = (Q @ D.T).cpu().numpy()
    run = {q["qid"]: rank_from_scores(S[k], doc_ids, 100) for k, q in enumerate(qs)}
    return evaluate(run, {q["qid"]: qrels[q["qid"]] for q in qs})["ndcg"]

In [ ]:
st_model, st_nombre = modelo_st()
antes = ndcg_st(st_model, q_test)
ds = Dataset.from_dict({"anchor": [p[0] for p in pares_poliglota], "positive": [txt_of[p[1]] for p in pares_poliglota]})
args = SentenceTransformerTrainingArguments(
    output_dir="st_out", num_train_epochs=1, per_device_train_batch_size=32, learning_rate=5e-4 if "fallback" in st_nombre else 2e-5,
    max_steps=150 if FAST_DEV_RUN else -1, batch_sampler=BatchSamplers.NO_DUPLICATES, report_to="none",
    logging_steps=50, save_strategy="no", seed=SEED, use_cpu=(device == "cpu"))
loss_mnrl = MultipleNegativesRankingLoss(st_model, scale=20.0)          # scale = 1/τ  -> τ = 0.05
SentenceTransformerTrainer(model=st_model, args=args, train_dataset=ds, loss=loss_mnrl).train()
despues = ndcg_st(st_model, q_test)
print(f"{st_nombre}: nDCG@10 test antes={antes:.3f} → después de MNRL={despues:.3f}")

> 👀 **Qué debes observar.** Unos pocos minutos de MNRL suben el nDCG@10 del modelo sobre las consultas de test (en la ejecución de referencia con el mini-BERT local: 0.40 → 0.58). Con `paraphrase-multilingual-MiniLM-L12-v2` real partes de un modelo ya bueno: usa LR baja (2e-5) y vigila que no empeore en otros idiomas (*catastrophic forgetting*).

In [ ]:
# ── LoRA para embedders: solo entrenamos matrices de rango r en las proyecciones de atención ──
from peft import LoraConfig, get_peft_model

base = st_model[0].auto_model                             # el Transformer dentro del SentenceTransformer
objetivo = [n for n in ("query", "value", "q_lin", "v_lin", "q_proj", "v_proj") if any(n in k for k, _ in base.named_modules())][:2]
cfg_lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=objetivo)
peft_model = get_peft_model(copy.deepcopy(base), cfg_lora)
entrenables = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in peft_model.parameters())
emb = sum(p.numel() for n, p in base.named_parameters() if "embeddings" in n)
print(f"módulos LoRA: {objetivo} | entrenables={entrenables:,} de {total:,} ({100 * entrenables / total:.2f} %) | embeddings del modelo={emb:,}")

> 👀 **Qué debes observar.** LoRA (Hu et al., 2021) entrena un porcentaje pequeño de parámetros (aquí ≈ 1–2 % con $r=8$ en `query`/`value`). En embedders multilingües pequeños buena parte de los parámetros está en la **tabla de embeddings** (módulo 09), que LoRA no toca: útil cuando quieres adaptar a un dominio sin alterar el vocabulario ni duplicar el modelo por cliente.

## 10. Estimación de cómputo para el modo completo
Colab cobra en **unidades de cómputo por hora** según la GPU; la tarifa vigente aparece en el panel de recursos de tu sesión (cambia con el tiempo, por eso no la fijamos aquí). Estima: **unidades ≈ horas × tarifa de tu GPU**.

In [ ]:
def estima_horas(n_pares, epocas, tokens_por_texto, params_no_embedding, tflops_efectivos):
    """FLOPs ≈ 6 · parámetros · tokens (forward + backward), con 2 textos por par (consulta y documento)."""
    tokens = n_pares * epocas * 2 * tokens_por_texto
    return 6 * params_no_embedding * tokens / (tflops_efectivos * 1e12) / 3600


# Ejemplo: 1M de pares, 1 época, 128 tokens, un encoder de ~21M parámetros fuera de la tabla de embeddings,
# y 20 TFLOP/s efectivos (orden de magnitud para una L4 con bf16 y buena utilización; mide el tuyo con el throughput real)
h = estima_horas(1_000_000, 1, 128, 21e6, 20)
print(f"≈ {h:.2f} h de cómputo puro; en la práctica multiplica por 2–3 (carga de datos, evaluación, utilización real).")
print("unidades de Colab ≈ horas_reales × (unidades/hora que muestra tu sesión para esa GPU)")

## Mapa de madurez (a oct-2026)

| Técnica | Madurez | Quién la usa (verificable) | Fuente |
|---|---|---|---|
| InfoNCE con negativos en el lote | 🟢 sólido | E5, GTE, BGE, Nomic Embed, Qwen3-Embedding (según sus informes técnicos) | Oord et al. 2018; Wang et al. 2022; Li et al. 2023; Xiao et al. 2023; Nussbaum et al. 2024; Zhang et al. 2025 |
| Negativos duros con BM25 / denso | 🟢 sólido | DPR (BM25), ANCE (denso) | Karpukhin et al. 2020; Xiong et al. 2020 |
| Filtros de falsos negativos (cross-encoder, TopK-PercPos) | 🟡 emergente → común | RocketQA; NV-Retriever (NVIDIA) | Qu et al. 2021; Moreira et al. 2024 |
| Pre-entrenamiento débil → ajuste fino | 🟢 sólido | E5 (CCPairs ≈ 270M pares), GTE, BGE | Wang et al. 2022; Li et al. 2023; Xiao et al. 2023 |
| SimCSE no supervisado | 🟡 útil sin etiquetas; superado por pares débiles | — | Gao et al. 2021 |
| Gradient Cache / lotes grandes | 🟢 sólido | `CachedMultipleNegativesRankingLoss` (sentence-transformers) | Gao, Zhang et al. 2021 |
| LoRA para embedders | 🟡 emergente | integración PEFT en sentence-transformers (`add_adapter`) | Hu et al. 2021 |
| Análisis alineación/uniformidad | 🟢 herramienta de diagnóstico estándar | SimCSE lo usa para analizar sus modelos | Wang & Isola 2020; Gao et al. 2021 |

## 🏭 En producción
- **Escala de datos.** E5 entrenó con ≈ 270M pares débiles (CCPairs) antes del ajuste supervisado; la calidad de un embedder moderno depende más de la **curación de pares** y del minado de negativos que de la arquitectura (ver informes de GTE, BGE/C-Pack, Nomic Embed, Qwen3-Embedding).
- **Falsos negativos a escala.** NV-Retriever (NVIDIA) documenta que el minado sin filtro introduce falsos negativos y propone filtros relativos al positivo (95 % en su configuración principal).
- **Memoria.** Los lotes grandes se consiguen con Gradient Cache (o con *data parallel* + reunir embeddings entre GPUs); en sentence-transformers: `CachedMultipleNegativesRankingLoss`.
- **Prefijos e instrucciones** (módulo 10) se fijan **durante** este entrenamiento: qué prefijo usa cada lado es parte del contrato del modelo.
- **Para Políglota:** el plan realista es ajustar un modelo multilingüe preentrenado con MNRL + negativos duros filtrados sobre pares propios (documentación técnica, docstrings↔código), y combinarlo con BM25 (módulo 13). El módulo 12 compara contra embedders comerciales y abiertos.

## 🧠 Secretos de la élite
1. **El candidato más parecido es el más sospechoso.** En nuestro laboratorio, los negativos de BM25 sin filtrar bajaron el nDCG de 0.88 a 0.84 y los filtrados lo subieron a 0.92 (prototipo de 300 pasos; con 200 pasos las diferencias se pierden en el ruido); en Políglota real el efecto fue pequeño (proyecto). Filtra siempre (umbral relativo al positivo, cross-encoder o deduplicación).
2. **InfoNCE se satura en $\log B$.** Lo comprobaste numéricamente: con lotes pequeños la señal está acotada; lotes grandes (o Gradient Cache, gradiente exacto) son una palanca de calidad, no solo de velocidad.
3. **Sin negativos hay colapso.** Alineación perfecta + uniformidad nula = modelo inútil; vigila $\mathcal L_{\text{unif}}$ durante el entrenamiento como alarma barata.
4. **$\tau$ es un hiperparámetro de primer orden** (0.05 SimCSE, 0.01 E5, `scale=20` en MNRL): un barrido de 4 valores cambió el nDCG de 0.49 a ≈ 1.00 en el laboratorio.
5. **Evita duplicados en el lote.** Dos pares que comparten documento o concepto se vuelven falsos negativos mutuos: `BatchSamplers.NO_DUPLICATES` o un muestreador por grupo.
6. **La pérdida de entrenamiento no es tu métrica.** En Políglota la pérdida bajaba mientras el nDCG de test se estancaba: evalúa en retrieval real cada pocos pasos.
7. **En benchmarks de ítem conocido, BM25 es durísimo.** Un denso pequeño entrenado desde cero no le gana; ajusta un preentrenado y combina (híbrido), y construye consultas no léxicas en tu *golden set* (módulo 15).

## ⚠️ Errores comunes
- Olvidar normalizar los embeddings cuando la pérdida asume coseno (o normalizar cuando el modelo se entrenó con producto escalar).
- Usar $\tau=1$ «por defecto» con coseno: los logits quedan en $[-1,1]$ y el softmax casi no distingue.
- Minar negativos duros sin filtrar falsos negativos.
- Lotes con duplicados (mismo documento positivo para dos consultas del lote).
- Evaluar con las mismas consultas con que se entrenó o eligió $\tau$.
- Implementar Gradient Cache con *dropout* sin reproducir el estado aleatorio entre los dos forwards.
- Ajustar un modelo preentrenado con LR alta y destruir su multilingüismo (*catastrophic forgetting*).

## Autoevaluación
<details><summary><b>1. (Cálculo)</b> Con un lote de $B=64$ y sin negativos extra, ¿cuál es el máximo de información mutua (en nats) que puede certificar InfoNCE? ¿Y con $B=4096$?</summary>

$\log 64\approx4.16$ nats; $\log 4096\approx8.32$ nats.
</details>
<details><summary><b>2. (Cálculo)</b> Una fila tiene similitudes coseno [0.80 (positivo), 0.78, 0.20] y $\tau=0.05$. ¿Qué probabilidad asigna el softmax al positivo y cuál es su pérdida?</summary>

Logits: 16, 15.6, 4. $p=e^{16}/(e^{16}+e^{15.6}+e^{4})\approx 1/(1+e^{-0.4})\approx0.599$; pérdida $-\log0.599\approx0.51$. Casi todo el gradiente lo genera el negativo de 0.78.
</details>
<details><summary><b>3. (Diagnóstico)</b> Tras entrenar, la pérdida es muy baja, la alineación ≈ 0 y la uniformidad ≈ 0, y el nDCG es pésimo. ¿Qué ha pasado y dónde buscas el bug?</summary>

Colapso: todos los embeddings son (casi) iguales. Con negativos de verdad en el denominador, un modelo colapsado tendría pérdida ≈ $\log B$ (todas las columnas empatan), así que una pérdida **muy baja** indica que los negativos no entran en la pérdida: pérdida solo de positivos (como `solo_positivos`), máscara que anula el denominador, o un lote en el que cada consulta solo «compite» consigo misma (p. ej. $B=1$). Una temperatura enorme también deja mala uniformidad, pero entonces la pérdida se queda cerca de $\log B$, no baja.
</details>
<details><summary><b>4. (Diagnóstico)</b> Al añadir negativos duros de BM25 el nDCG baja. ¿Qué compruebas antes de descartarlos?</summary>

La tasa de falsos negativos: revisa a mano una muestra del top de BM25 y aplica un filtro relativo al positivo (TopK-PercPos) o un cross-encoder; prueba también a introducirlos después de un calentamiento con negativos aleatorios.
</details>
<details><summary><b>5. (Transferencia)</b> En tu *two-tower* de recsys con negativos en el lote, los ítems populares salen demasiado poco recomendados. ¿Qué tiene que ver con este módulo y qué corrección aplicas?</summary>

Los populares aparecen más a menudo como negativos en el lote y se penalizan de más (sesgo de muestreo). Corrección logQ: restar $\log Q(\text{ítem})$ (probabilidad de muestreo) al logit, como en `recsys-course/08_deep_retrieval_ann`.
</details>
<details><summary><b>6.</b> ¿Por qué Gradient Cache da el gradiente exacto y no una aproximación?</summary>

Porque la pérdida solo depende de los parámetros a través de las representaciones: $\partial\mathcal L/\partial\theta=\sum_i(\partial\mathcal L/\partial\mathbf z_i)(\partial\mathbf z_i/\partial\theta)$. El primer factor se calcula con todo el lote; el segundo, por trozos.
</details>
<details><summary><b>7. (Transferencia)</b> Políglota quiere añadir búsqueda «docstring → código». ¿Qué pares débiles y qué negativos duros usarías?</summary>

Pares: (primera frase del docstring, función sin docstring) y (nombre de la función partido en palabras, cuerpo). Negativos duros: funciones del mismo módulo/repositorio con identificadores parecidos, filtradas para no incluir duplicados o *forks* del mismo código.
</details>

## Referencias
- van den Oord, A., Li, Y., & Vinyals, O. (2018). *Representation Learning with Contrastive Predictive Coding.* Preprint. [arXiv:1807.03748](https://arxiv.org/abs/1807.03748)
- Poole, B., Ozair, S., van den Oord, A., et al. (2019). *On Variational Bounds of Mutual Information.* ICML 2019. [arXiv:1905.06922](https://arxiv.org/abs/1905.06922)
- McAllester, D., & Stratos, K. (2020). *Formal Limitations on the Measurement of Mutual Information.* AISTATS 2020. [arXiv:1811.04251](https://arxiv.org/abs/1811.04251)
- Tschannen, M., Djolonga, J., Rubenstein, P. K., et al. (2020). *On Mutual Information Maximization for Representation Learning.* ICLR 2020. [arXiv:1907.13625](https://arxiv.org/abs/1907.13625)
- Wang, T., & Isola, P. (2020). *Understanding Contrastive Representation Learning through Alignment and Uniformity on the Hypersphere.* ICML 2020. [arXiv:2005.10242](https://arxiv.org/abs/2005.10242)
- Wang, F., & Liu, H. (2021). *Understanding the Behaviour of Contrastive Loss.* CVPR 2021. [arXiv:2012.09740](https://arxiv.org/abs/2012.09740)
- Chen, T., Kornblith, S., Norouzi, M., & Hinton, G. (2020). *A Simple Framework for Contrastive Learning of Visual Representations* (SimCLR). ICML 2020. [arXiv:2002.05709](https://arxiv.org/abs/2002.05709)
- Henderson, M., Al-Rfou, R., Strope, B., et al. (2017). *Efficient Natural Language Response Suggestion for Smart Reply.* Preprint. [arXiv:1705.00652](https://arxiv.org/abs/1705.00652)
- Gao, T., Yao, X., & Chen, D. (2021). *SimCSE: Simple Contrastive Learning of Sentence Embeddings.* EMNLP 2021. [arXiv:2104.08821](https://arxiv.org/abs/2104.08821)
- Karpukhin, V., Oğuz, B., Min, S., et al. (2020). *Dense Passage Retrieval for Open-Domain Question Answering.* EMNLP 2020. [arXiv:2004.04906](https://arxiv.org/abs/2004.04906)
- Xiong, L., Xiong, C., Li, Y., et al. (2020). *Approximate Nearest Neighbor Negative Contrastive Learning for Dense Text Retrieval* (ANCE). ICLR 2021. [arXiv:2007.00808](https://arxiv.org/abs/2007.00808)
- Qu, Y., Ding, Y., Liu, J., et al. (2021). *RocketQA: An Optimized Training Approach to Dense Passage Retrieval for Open-Domain Question Answering.* NAACL 2021. [arXiv:2010.08191](https://arxiv.org/abs/2010.08191)
- Hofstätter, S., Lin, S.-C., Yang, J.-H., et al. (2021). *Efficiently Teaching an Effective Dense Retriever with Balanced Topic Aware Sampling* (TAS-B). SIGIR 2021. [arXiv:2104.06967](https://arxiv.org/abs/2104.06967)
- Moreira, G. de S. P., Osmulski, R., Xu, M., et al. (2024). *NV-Retriever: Improving text embedding models with effective hard-negative mining.* Preprint. [arXiv:2407.15831](https://arxiv.org/abs/2407.15831)
- Robinson, J., Chuang, C.-Y., Sra, S., & Jegelka, S. (2021). *Contrastive Learning with Hard Negative Samples.* ICLR 2021. [arXiv:2010.04592](https://arxiv.org/abs/2010.04592)
- Gao, L., Zhang, Y., Han, J., & Callan, J. (2021). *Scaling Deep Contrastive Learning Batch Size under Memory Limited Setup* (Gradient Cache). RepL4NLP 2021. [arXiv:2101.06983](https://arxiv.org/abs/2101.06983)
- Wang, L., Yang, N., Huang, X., et al. (2022). *Text Embeddings by Weakly-Supervised Contrastive Pre-training* (E5). Preprint. [arXiv:2212.03533](https://arxiv.org/abs/2212.03533)
- Li, Z., Zhang, X., Zhang, Y., et al. (2023). *Towards General Text Embeddings with Multi-stage Contrastive Learning* (GTE). Preprint. [arXiv:2308.03281](https://arxiv.org/abs/2308.03281)
- Xiao, S., Liu, Z., Zhang, P., et al. (2023). *C-Pack: Packed Resources For General Chinese Embeddings* (BGE). SIGIR 2024. [arXiv:2309.07597](https://arxiv.org/abs/2309.07597)
- Nussbaum, Z., Morris, J. X., Duderstadt, B., et al. (2024). *Nomic Embed: Training a Reproducible Long Context Text Embedder.* TMLR. [arXiv:2402.01613](https://arxiv.org/abs/2402.01613)
- Zhang, Y., Li, M., Long, D., et al. (2025). *Qwen3 Embedding: Advancing Text Embedding and Reranking Through Foundation Models.* Preprint. [arXiv:2506.05176](https://arxiv.org/abs/2506.05176)
- Hu, E. J., Shen, Y., Wallis, P., et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models.* ICLR 2022. [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)
- Librería: sentence-transformers (`MultipleNegativesRankingLoss`, `CachedMultipleNegativesRankingLoss`, `BatchSamplers.NO_DUPLICATES`, `mine_hard_negatives`): https://pypi.org/project/sentence-transformers/

## Siguiente módulo
**12 · Embedders modernos**: familias E5/BGE/GTE/Qwen3-Embedding/EmbeddingGemma, LLMs como embedders, Matryoshka y cuantización, y cómo elegir uno para Políglota frente al que acabas de entrenar.